In [0]:
# --------------------------------------------------
# 1. GET CURRENT RUN
# --------------------------------------------------

running_runs = spark.sql("""
    SELECT
        run_id,
        batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = 'orders_pipeline'
      AND status = 'RUNNING'
""").collect()

if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, found {len(running_runs)}"
    )

run_id = running_runs[0]["run_id"]
batch_id = running_runs[0]["batch_id"]

print(f"Run ID:   {run_id}")
print(f"Batch ID: {batch_id}")


try:

    # --------------------------------------------------
    # 2. COUNTS BEFORE PROCESSING
    # --------------------------------------------------

    silver_before = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.silver.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    rejected_before = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.control.rejected_orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]


    # --------------------------------------------------
    # 3. BRONZE -> SILVER
    # --------------------------------------------------

    spark.sql(f"""
    INSERT INTO workspace.silver.orders (
        order_id,
        customer_id,
        amount_raw,
        amount,
        status,
        order_date_raw,
        order_date,
        last_updated,
        event_id,
        batch_id,
        source_file,
        load_timestamp,
        dq_amount,
        dq_customer_id,
        dq_overall
    )

    WITH cleaned AS (

        SELECT
            b.*,

            CASE
                WHEN amount IS NULL THEN NULL

                WHEN UPPER(TRIM(amount)) = 'N/A'
                THEN NULL

                -- number only / number + currency
                WHEN TRIM(amount)
                     RLIKE '^-?[0-9]+(\\.[0-9]+)?([ ]+[A-Za-z]+)?$'
                THEN TRY_CAST(
                    REGEXP_EXTRACT(
                        TRIM(amount),
                        '^(-?[0-9]+(?:\\.[0-9]+)?)',
                        1
                    )
                    AS DECIMAL(10,2)
                )

                -- currency + number
                WHEN TRIM(amount)
                     RLIKE '^[A-Za-z]+[ ]+-?[0-9]+(\\.[0-9]+)?$'
                THEN TRY_CAST(
                    REGEXP_EXTRACT(
                        TRIM(amount),
                        '(-?[0-9]+(?:\\.[0-9]+)?)$',
                        1
                    )
                    AS DECIMAL(10,2)
                )

                ELSE NULL
            END AS amount_clean,


            COALESCE(
                TRY_TO_DATE(TRIM(order_date), 'yyyy-MM-dd'),
                TRY_TO_DATE(TRIM(order_date), 'dd/MM/yyyy'),
                TRY_TO_DATE(TRIM(order_date), 'dd-MM-yyyy'),
                TRY_TO_DATE(TRIM(order_date), 'yyyy/MM/dd')
            ) AS order_date_clean

        FROM workspace.bronze.orders b

        WHERE b.batch_id = '{batch_id}'
    ),


    prepared AS (

        SELECT
            *,

            CASE
                WHEN amount IS NULL THEN 1
                WHEN UPPER(TRIM(amount)) = 'N/A' THEN 1
                WHEN amount_clean IS NOT NULL THEN 1
                ELSE 0
            END AS dq_amount,

            CASE
                WHEN customer_id IS NULL THEN 1
                WHEN TRY_CAST(customer_id AS INT) IS NOT NULL THEN 1
                ELSE 0
            END AS dq_customer_id,

            CASE
                WHEN order_date IS NULL THEN 0
                WHEN order_date_clean IS NOT NULL THEN 1
                ELSE 0
            END AS dq_order_date

        FROM cleaned
    )

    SELECT
        TRY_CAST(order_id AS INT) AS order_id,
        TRY_CAST(customer_id AS INT) AS customer_id,

        amount AS amount_raw,
        amount_clean AS amount,

        UPPER(TRIM(status)) AS status,

        order_date AS order_date_raw,
        order_date_clean AS order_date,

        TRY_CAST(last_updated AS TIMESTAMP) AS last_updated,
        TRY_CAST(event_id AS INT) AS event_id,

        batch_id,
        source_file,
        load_timestamp,

        dq_amount,
        dq_customer_id,

        CASE
            WHEN TRY_CAST(order_id AS INT) IS NULL THEN 0
            WHEN dq_amount = 0 THEN 0
            WHEN dq_order_date = 0 THEN 0
            WHEN dq_customer_id = 0 THEN 0
            ELSE 1
        END AS dq_overall

    FROM prepared p

    WHERE NOT EXISTS (
        SELECT 1
        FROM workspace.silver.orders s
        WHERE s.batch_id = p.batch_id
          AND s.event_id = TRY_CAST(p.event_id AS INT)
    )

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY batch_id, event_id
        ORDER BY
            TRY_CAST(last_updated AS TIMESTAMP) DESC,
            load_timestamp DESC
    ) = 1
    """)

    print("Silver load successful.")


    # --------------------------------------------------
    # 4. REJECTED
    # --------------------------------------------------

    spark.sql(f"""
    INSERT INTO workspace.control.rejected_orders (
        order_id,
        customer_id,
        amount_raw,
        amount_clean,
        status,
        order_date_raw,
        order_date_clean,
        last_updated,
        event_id,
        batch_id,
        source_file,
        rejection_reason,
        rejected_at
    )

    SELECT
        CAST(o.order_id AS STRING),
        CAST(o.customer_id AS STRING),

        o.amount_raw,
        o.amount,

        o.status,

        o.order_date_raw,
        o.order_date,

        CAST(o.last_updated AS STRING),
        CAST(o.event_id AS STRING),

        o.batch_id,
        o.source_file,

        CASE
            WHEN o.dq_amount = 0
                THEN 'INVALID_AMOUNT'

            WHEN o.dq_customer_id = 0
                THEN 'INVALID_CUSTOMER_ID'

            ELSE 'DQ_FAILED'
        END,

        CURRENT_TIMESTAMP()

    FROM workspace.silver.orders o

    WHERE o.batch_id = '{batch_id}'
      AND o.dq_overall = 0

      AND NOT EXISTS (
          SELECT 1
          FROM workspace.control.rejected_orders ro
          WHERE ro.batch_id = o.batch_id
            AND ro.event_id = CAST(o.event_id AS STRING)
      )
    """)

    print("Rejected rows processed.")


    # --------------------------------------------------
    # 5. COUNTS AFTER PROCESSING
    # --------------------------------------------------

    silver_after = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.silver.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    rejected_after = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.control.rejected_orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]


    silver_rows = silver_after - silver_before
    rejected_rows = rejected_after - rejected_before


    # --------------------------------------------------
    # 6. SILVER MONITORING METRICS
    # --------------------------------------------------

    # Number of excess duplicate event rows in Bronze
    # Example:
    # event_id appears 2 times -> 1 row removed
    # event_id appears 3 times -> 2 rows removed

    duplicate_event_rows_removed = spark.sql(f"""
        SELECT
            COALESCE(SUM(cnt - 1), 0) AS duplicate_rows_removed

        FROM (
            SELECT
                batch_id,
                event_id,
                COUNT(*) AS cnt

            FROM workspace.bronze.orders

            WHERE batch_id = '{batch_id}'

            GROUP BY
                batch_id,
                event_id

            HAVING COUNT(*) > 1
        )
    """).first()["duplicate_rows_removed"]


    # Number of valid order_ids that have more than one
    # event/version in Silver.
    #
    # These are not rejected.
    # Gold later collapses them to the latest business state.

    multi_version_order_count = spark.sql(f"""
        SELECT
            COUNT(*) AS multi_version_orders

        FROM (
            SELECT
                order_id

            FROM workspace.silver.orders

            WHERE batch_id = '{batch_id}'
              AND dq_overall = 1

            GROUP BY order_id

            HAVING COUNT(*) > 1
        )
    """).first()["multi_version_orders"]


    # --------------------------------------------------
    # 7. UPDATE AUDIT
    # --------------------------------------------------

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            silver_rows = {silver_rows},
            rejected_rows = {rejected_rows},
            duplicate_event_rows_removed = {duplicate_event_rows_removed},
            multi_version_order_count = {multi_version_order_count}

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)


    # --------------------------------------------------
    # 8. SUMMARY
    # --------------------------------------------------

    print("----------------------------------")
    print("SILVER PROCESSING COMPLETE")
    print("----------------------------------")
    print(f"Run ID:                       {run_id}")
    print(f"Batch ID:                     {batch_id}")
    print(f"Silver inserted this run:     {silver_rows}")
    print(f"Rejected inserted this run:   {rejected_rows}")
    print(f"Duplicate rows removed:       {duplicate_event_rows_removed}")
    print(f"Multi-version orders:         {multi_version_order_count}")
    print("----------------------------------")


except Exception as e:

    error_message = str(e).replace("'", "''")[:4000]

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            end_timestamp = CURRENT_TIMESTAMP(),
            status = 'FAILED',
            error_message = '{error_message}'

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)

    print("Silver processing FAILED.")
    print(error_message)

    raise

Run ID:   596f9a73-298f-478d-94d2-c240a58bf207
Batch ID: batch_006
Silver load successful.
Rejected rows processed.
----------------------------------
SILVER PROCESSING COMPLETE
----------------------------------
Run ID:                    596f9a73-298f-478d-94d2-c240a58bf207
Batch ID:                  batch_006
Silver inserted this run:  406
Rejected inserted this run:2
----------------------------------


In [0]:
%sql
select * from control.rejected_orders
where batch_id='batch_006'

order_id,customer_id,amount_raw,amount_clean,status,order_date_raw,order_date_clean,last_updated,event_id,batch_id,source_file,rejection_reason,rejected_at
103300,5100,12AB34,null,PENDING,2026-09-02,2026-09-02,2026-09-02 12:00:00,12001,batch_006,orders_batch_006.csv,INVALID_AMOUNT,2026-09-02T18:46:15.155Z
103301,5101,555.55,555.55,COMPLETED,2026-99-99,null,2026-09-02 12:01:00,12002,batch_006,orders_batch_006.csv,DQ_FAILED,2026-09-02T18:46:15.155Z
